# 🌲 ComexStat Monitor — Pine Chemicals
### RJ Comércio & Alliance Resin Partners

**Fonte:** [ComexStat MDIC](https://comexstat.mdic.gov.br)  
**API:** `POST https://api-comexstat.mdic.gov.br/general`

---
**NCMs monitorados:**
| NCM | Descrição |
|---|---|
| 3806.10.00 | Colofônia / Gum Rosin |
| 3805.10.10 | Terebintina de Goma |
| 3805.10.90 | Outras Terebintinas |
| 3806.20.00 | Sais de ácidos resinosos |
| 3806.30.00 | Gomas éster |

---
▶ **Execute célula por célula** ou use `Runtime > Run all`

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 1 — Instalação de dependências
# ═══════════════════════════════════════════════════════════
!pip install -q openpyxl requests
print('✅ Dependências OK')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 2 — ⚙️ CONFIGURAÇÕES (edite aqui)
# ═══════════════════════════════════════════════════════════

ANO_INICIAL = '2022'   # ← altere
ANO_FINAL   = '2025'   # ← altere
FLUXO       = 'export' # 'export' ou 'import'

# NCMs a consultar — remova ou adicione conforme necessidade
NCMS = {
    "13019090": "Goma Resina",
    "38051000": "Terebintina",
    "38061000": "Breu",
    "38069011": "Derivados1",
    "38069019": "Derivados2",
    "38069090": "Derivados3",
}

print(f'📋 Configuração:')
print(f'   Período : {ANO_INICIAL} – {ANO_FINAL}')
print(f'   Fluxo   : {FLUXO.upper()}')
print(f'   NCMs    : {list(NCMS.keys())}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 3 — Funções de consulta à API ComexStat
# ═══════════════════════════════════════════════════════════
import requests
import json
import time

API_URL = 'https://api-comexstat.mdic.gov.br/general'

MAX_RETRIES = 5
BACKOFF_BASE = 15  # segundos

def buscar_comexstat(ncm, ano_ini, ano_fim, fluxo='export'):
    payload = {
        'flow':        fluxo,
        'monthDetail': True,
        'period': {
            'from': f'{ano_ini}-01',
            'to':   f'{ano_fim}-12'
        },
        'filters': [
            {'filter': 'ncm', 'values': [ncm]}
        ],
        'details': ['country', 'ncm'],
        'metrics': ['metricFOB', 'metricKG', 'metricStatistic']
    }
    headers = {
        'Content-Type': 'application/json',
        'Accept':       'application/json',
    }
    import urllib3; urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    for tentativa in range(1, MAX_RETRIES + 1):
        resp = requests.post(API_URL, json=payload, headers=headers, timeout=30, verify=False)
        if resp.status_code == 429:
            espera = BACKOFF_BASE * (2 ** (tentativa - 1))
            print(f'\n      ⏳ 429 — tentativa {tentativa}/{MAX_RETRIES}, aguardando {espera}s...', end=' ', flush=True)
            time.sleep(espera)
            continue
        resp.raise_for_status()
        data = resp.json()
        registros = (
            data.get('data', {}).get('list', []) or
            data.get('list', [])
        )
        return registros

    raise requests.exceptions.HTTPError(
        f'Rate limit persistente apos {MAX_RETRIES} tentativas',
        response=resp
    )


def normalizar_registro(reg):
    def to_float(v):
        try:
            return float(str(v or 0).replace(',', '.'))
        except:
            return 0.0
    ano   = reg.get('year',  reg.get('coAno',  ''))
    mes   = reg.get('monthNumber', reg.get('month', reg.get('coMes', '')))
    pais  = reg.get('country', reg.get('countryNamePt', reg.get('noPaispt', 'N/D')))
    fob   = to_float(reg.get('metricFOB',        reg.get('vlFob',      0)))
    kg    = to_float(reg.get('metricKG',          reg.get('kgLiquido',  0)))
    qtd   = to_float(reg.get('metricStatistic',   reg.get('qtEstat',    0)))
    preco = fob / kg if kg > 0 else 0.0
    return {
        'ano': ano, 'mes': mes, 'pais': pais,
        'fob_usd': fob, 'kg_liquido': kg,
        'preco_usd_kg': preco, 'qtd_estatistica': qtd
    }

print('✅ Funções de API carregadas (com retry automatico)')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 4 — 🔄 Busca dos dados
# ═══════════════════════════════════════════════════════════
from IPython.display import display, HTML

todos_dados = {}
erros = []
INTERVALO_ENTRE_NCMS = 20

for idx, (ncm, descricao) in enumerate(NCMS.items()):
    if idx > 0:
        print(f'   💤 Aguardando {INTERVALO_ENTRE_NCMS}s entre requisicoes...')
        time.sleep(INTERVALO_ENTRE_NCMS)

    print(f'\n⏳ [{ncm}] {descricao}...', end=' ', flush=True)
    try:
        registros_brutos = buscar_comexstat(ncm, ANO_INICIAL, ANO_FINAL, FLUXO)
        todos_dados[ncm] = [normalizar_registro(r) for r in registros_brutos]
        print(f'✅ {len(todos_dados[ncm])} registros')
    except requests.exceptions.HTTPError as e:
        print(f'❌ HTTP {e.response.status_code}')
        print(f'   Resposta: {e.response.text[:200]}')
        erros.append(ncm)
        todos_dados[ncm] = []
    except Exception as e:
        print(f'❌ {e}')
        erros.append(ncm)
        todos_dados[ncm] = []

total = sum(len(v) for v in todos_dados.values())
print(f'\n{"="*50}')
print(f'📊 Total de registros coletados: {total}')
if erros:
    print(f'⚠️  NCMs com erro: {erros}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 5 — 📊 Preview dos dados em tabela
# ═══════════════════════════════════════════════════════════
import pandas as pd
from IPython.display import display

for ncm, descricao in NCMS.items():
    dados = todos_dados.get(ncm, [])
    if not dados:
        print(f'\n⚠️  [{ncm}] Sem dados')
        continue

    df = pd.DataFrame(dados)
    df = df.sort_values(['ano', 'mes'], ascending=False)

    # Formata para exibição
    df_display = df.copy()
    df_display['fob_usd']      = df_display['fob_usd'].map('{:,.2f}'.format)
    df_display['kg_liquido']   = df_display['kg_liquido'].map('{:,.0f}'.format)
    df_display['preco_usd_kg'] = df_display['preco_usd_kg'].map('{:.4f}'.format)

    print(f'\n🌲 {descricao} (NCM {ncm[:4]}.{ncm[4:6]}.{ncm[6:]}) — {len(df)} registros')

    # Resumo por ano
    resumo = df.groupby('ano').agg(
        FOB_USD=('fob_usd', 'sum'),
        KG=('kg_liquido', 'sum'),
        Paises=('pais', 'nunique')
    ).reset_index()
    resumo['USD_por_kg'] = resumo['FOB_USD'] / resumo['KG'].replace(0, float('nan'))
    resumo['FOB_USD'] = resumo['FOB_USD'].map('${:,.0f}'.format)
    resumo['KG'] = resumo['KG'].map('{:,.0f}'.format)
    resumo['USD_por_kg'] = resumo['USD_por_kg'].map('{:.4f}'.format)
    print('  Resumo por ano:')
    display(resumo)

    print('  Últimos registros:')
    display(df_display.head(10))

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 6 — 📈 Gráficos de preço médio e volume
# ═══════════════════════════════════════════════════════════
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

NCM_BREU = '38061000'
dados_breu = todos_dados.get(NCM_BREU, [])

if dados_breu:
    df_b = pd.DataFrame(dados_breu)
    df_b['ano_mes'] = df_b['ano'].astype(str) + '-' + df_b['mes'].astype(str).str.zfill(2)

    # Agrega por mês
    mensal = df_b.groupby('ano_mes').agg(
        fob=('fob_usd', 'sum'),
        kg=('kg_liquido', 'sum')
    ).reset_index().sort_values('ano_mes')
    mensal['preco'] = mensal['fob'] / mensal['kg'].replace(0, float('nan'))
    mensal = mensal.dropna(subset=['preco'])

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fig.patch.set_facecolor('#F8FFF8')

    # Cor RJ Comércio
    COR_VERDE = '#2D6A4F'
    COR_CLARO = '#52B788'

    # Gráfico 1 — Preço médio USD/kg
    ax1 = axes[0]
    ax1.set_facecolor('#F0FAF4')
    ax1.plot(mensal['ano_mes'], mensal['preco'], color=COR_VERDE,
             linewidth=2.5, marker='o', markersize=4)
    ax1.fill_between(mensal['ano_mes'], mensal['preco'],
                     alpha=0.15, color=COR_VERDE)
    ax1.set_title('Preço Médio FOB — Colofônia / Gum Rosin (USD/kg)',
                  fontsize=12, fontweight='bold', color='#1A4731', pad=10)
    ax1.set_ylabel('USD / kg', color='#1A4731')
    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax1.grid(True, alpha=0.3, color='#2D6A4F')
    tick_step = max(1, len(mensal) // 12)
    ax1.set_xticks(range(0, len(mensal), tick_step))
    ax1.set_xticklabels(mensal['ano_mes'].iloc[::tick_step],
                        rotation=45, ha='right', fontsize=8)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    # Gráfico 2 — Volume KG
    ax2 = axes[1]
    ax2.set_facecolor('#F0FAF4')
    ax2.bar(range(len(mensal)), mensal['kg'] / 1000,
            color=COR_CLARO, alpha=0.8, width=0.7)
    ax2.set_title('Volume Exportado — Colofônia / Gum Rosin (toneladas)',
                  fontsize=12, fontweight='bold', color='#1A4731', pad=10)
    ax2.set_ylabel('Toneladas', color='#1A4731')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}t'))
    ax2.set_xticks(range(0, len(mensal), tick_step))
    ax2.set_xticklabels(mensal['ano_mes'].iloc[::tick_step],
                        rotation=45, ha='right', fontsize=8)
    ax2.grid(True, alpha=0.3, axis='y', color='#2D6A4F')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    plt.suptitle('RJ Comércio & Alliance Resin Partners — ComexStat Monitor',
                 fontsize=10, color='#888888', y=1.01)
    plt.tight_layout()
    plt.savefig('grafico_pine_chemicals.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Gráfico salvo: grafico_pine_chemicals.png')
else:
    print('⚠️  Sem dados de breu para gráfico')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 7 — 📄 Geração do Excel
# ═══════════════════════════════════════════════════════════
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.chart import LineChart, Reference
from openpyxl.chart.series import DataPoint
from datetime import datetime

VERDE_ESCURO = '1A4731'
VERDE_MEDIO  = '2D6A4F'
VERDE_CLARO  = 'D8F3DC'
BRANCO       = 'FFFFFF'
CINZA        = 'F5F5F5'

def cell(ws, row, col, value=None, bold=False, bg=None, fg='333333',
         align='center', size=10, num_fmt=None):
    c = ws.cell(row=row, column=col)
    if value is not None:
        c.value = value
    c.font = Font(name='Arial', bold=bold, color=fg, size=size)
    c.alignment = Alignment(horizontal=align, vertical='center', wrap_text=True)
    if bg:
        c.fill = PatternFill('solid', fgColor=bg)
    if num_fmt:
        c.number_format = num_fmt
    return c


def criar_aba_dados(wb, nome_aba, dados, ncm, descricao):
    ws = wb.create_sheet(nome_aba)
    ws.sheet_view.showGridLines = False
    ws.column_dimensions['A'].width = 2
    ws.column_dimensions['B'].width = 8
    ws.column_dimensions['C'].width = 6
    ws.column_dimensions['D'].width = 30
    ws.column_dimensions['E'].width = 16
    ws.column_dimensions['F'].width = 16
    ws.column_dimensions['G'].width = 12
    ws.column_dimensions['H'].width = 12

    ws.merge_cells('B1:H2')
    c = ws['B1']
    ncm_fmt = f'{ncm[:4]}.{ncm[4:6]}.{ncm[6:]}'
    c.value = f'{descricao}  —  NCM {ncm_fmt}  |  {len(dados)} registros'
    c.font = Font(name='Arial', bold=True, color=BRANCO, size=11)
    c.fill = PatternFill('solid', fgColor=VERDE_ESCURO)
    c.alignment = Alignment(horizontal='center', vertical='center')

    hdrs = ['Ano', 'Mes', 'Pais Destino', 'FOB (USD)', 'KG Liquido', 'USD/kg', 'Qtd Estat.']
    fmts = [None, None, None, '#,##0.00', '#,##0', '0.0000', '#,##0']
    alns = ['center','center','left','right','right','right','right']

    for j, h in enumerate(hdrs, 2):
        cell(ws, 4, j, h, bold=True, bg=VERDE_ESCURO, fg=BRANCO, size=9)
    ws.row_dimensions[4].height = 18

    dados_sorted = sorted(dados, key=lambda x: (x['ano'], x['mes']))
    for i, reg in enumerate(dados_sorted, 5):
        bg = CINZA if i % 2 == 0 else BRANCO
        vals = [reg['ano'], reg['mes'], reg['pais'],
                reg['fob_usd'], reg['kg_liquido'],
                reg['preco_usd_kg'], reg['qtd_estatistica']]
        for j, (v, fmt, aln) in enumerate(zip(vals, fmts, alns), 2):
            cell(ws, i, j, v, bg=bg, align=aln, num_fmt=fmt)
        ws.row_dimensions[i].height = 15

    return ws, len(dados_sorted) + 4  # última linha de dados


def criar_aba_preco_medio(wb, todos_dados):
    ws = wb.create_sheet('Preco_Mensal')
    ws.sheet_view.showGridLines = False
    for col, w in zip('ABCDEFG', [2, 10, 6, 16, 16, 12, 2]):
        ws.column_dimensions[col].width = w

    ws.merge_cells('B1:F2')
    c = ws['B1']
    c.value = 'PRECO MEDIO FOB (USD/kg) — COLOFONIA / GUM ROSIN'
    c.font = Font(name='Arial', bold=True, color=BRANCO, size=11)
    c.fill = PatternFill('solid', fgColor=VERDE_ESCURO)
    c.alignment = Alignment(horizontal='center', vertical='center')

    hdrs = ['Ano', 'Mes', 'FOB Total (USD)', 'KG Total', 'USD/kg Medio']
    for j, h in enumerate(hdrs, 2):
        cell(ws, 4, j, h, bold=True, bg=VERDE_ESCURO, fg=BRANCO, size=9)
    ws.row_dimensions[4].height = 18

    dados_breu = todos_dados.get('38061000', [])
    agg = {}
    for r in dados_breu:
        k = (r['ano'], r['mes'])
        if k not in agg:
            agg[k] = {'fob': 0, 'kg': 0}
        agg[k]['fob'] += r['fob_usd']
        agg[k]['kg']  += r['kg_liquido']

    for i, ((ano, mes), v) in enumerate(sorted(agg.items()), 5):
        bg = CINZA if i % 2 == 0 else BRANCO
        preco = v['fob'] / v['kg'] if v['kg'] > 0 else 0
        cell(ws, i, 2, ano,     bg=bg)
        cell(ws, i, 3, mes,     bg=bg)
        cell(ws, i, 4, v['fob'], bg=bg, align='right', num_fmt='#,##0.00')
        cell(ws, i, 5, v['kg'],  bg=bg, align='right', num_fmt='#,##0')
        cell(ws, i, 6, preco,   bg=bg, align='right', num_fmt='0.0000')
        ws.row_dimensions[i].height = 15

    ultima_linha = 4 + len(agg)

    # Gráfico de linha — Preço médio
    if ultima_linha > 5:
        chart = LineChart()
        chart.title = 'Preco Medio FOB — Colofonia (USD/kg)'
        chart.style = 10
        chart.y_axis.title = 'USD/kg'
        chart.x_axis.title = 'Periodo'
        chart.height = 12
        chart.width  = 22

        data_ref = Reference(ws, min_col=6, min_row=4, max_row=ultima_linha)
        cats_ref = Reference(ws, min_col=2, min_row=5, max_row=ultima_linha)

        chart.add_data(data_ref, titles_from_data=True)
        chart.set_categories(Reference(ws, min_col=2, max_col=3,
                                        min_row=5, max_row=ultima_linha))
        ws.add_chart(chart, 'B' + str(ultima_linha + 3))

    return ws


# ── Monta o workbook ───────────────────────────────────────
wb = Workbook()
ws_resumo = wb.active
ws_resumo.title = 'Resumo'
ws_resumo.sheet_view.showGridLines = False

# Banner
ws_resumo.merge_cells('B1:G2')
c = ws_resumo['B1']
c.value = 'PINE CHEMICALS — COMEXSTAT MONITOR | RJ Comercio & Alliance Resin Partners'
c.font = Font(name='Arial', bold=True, color=BRANCO, size=12)
c.fill = PatternFill('solid', fgColor=VERDE_ESCURO)
c.alignment = Alignment(horizontal='center', vertical='center')
ws_resumo.row_dimensions[1].height = 22
ws_resumo.row_dimensions[2].height = 22

ws_resumo.merge_cells('B3:G3')
c = ws_resumo['B3']
c.value = (f'Fonte: ComexStat MDIC  |  Fluxo: {FLUXO.upper()}  |  '
           f'Periodo: {ANO_INICIAL}–{ANO_FINAL}  |  '
           f'Gerado em: {datetime.now().strftime("%d/%m/%Y %H:%M")}')
c.font = Font(name='Arial', size=9, italic=True, color=VERDE_ESCURO)
c.fill = PatternFill('solid', fgColor=VERDE_CLARO)
c.alignment = Alignment(horizontal='center', vertical='center')

# Tabela de totais no Resumo
row = 5
for col_w, col_letter in zip([2,12,14,28,16,16,14], 'ABCDEFG'):
    ws_resumo.column_dimensions[col_letter].width = col_w

hdrs_res = ['NCM', 'Produto', 'Registros', 'FOB Total (USD)', 'KG Total', 'USD/kg Medio']
for j, h in enumerate(hdrs_res, 2):
    cell(ws_resumo, row, j, h, bold=True, bg=VERDE_ESCURO, fg=BRANCO, size=9)
ws_resumo.row_dimensions[row].height = 18
row += 1

for i, (ncm, descricao) in enumerate(NCMS.items()):
    dados = todos_dados.get(ncm, [])
    bg = CINZA if i % 2 == 0 else BRANCO
    ncm_fmt = f'{ncm[:4]}.{ncm[4:6]}.{ncm[6:]}'
    fob_total = sum(r['fob_usd'] for r in dados)
    kg_total  = sum(r['kg_liquido'] for r in dados)
    preco_med = fob_total / kg_total if kg_total > 0 else 0

    cell(ws_resumo, row, 2, ncm_fmt,       bg=bg)
    cell(ws_resumo, row, 3, descricao,     bg=bg, align='left')
    cell(ws_resumo, row, 4, len(dados),    bg=bg)
    cell(ws_resumo, row, 5, fob_total,     bg=bg, align='right', num_fmt='#,##0.00')
    cell(ws_resumo, row, 6, kg_total,      bg=bg, align='right', num_fmt='#,##0')
    cell(ws_resumo, row, 7, preco_med,     bg=bg, align='right', num_fmt='0.0000')
    ws_resumo.row_dimensions[row].height = 16
    row += 1

# Abas por NCM
for ncm, descricao in NCMS.items():
    dados = todos_dados.get(ncm, [])
    if dados:
        nome_aba = f'{ncm[:4]}.{ncm[4:6]}.{ncm[6:]}'
        criar_aba_dados(wb, nome_aba, dados, ncm, descricao)

# Aba preço médio
criar_aba_preco_medio(wb, todos_dados)

# Salva
NOME_ARQUIVO = f'comexstat_pine_chemicals_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'
wb.save(NOME_ARQUIVO)
print(f'✅ Excel gerado: {NOME_ARQUIVO}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 8 — ⬇️ Download do Excel
# ═══════════════════════════════════════════════════════════
from google.colab import files

files.download(NOME_ARQUIVO)
print(f'📥 Download iniciado: {NOME_ARQUIVO}')

# Opcional: baixar também o gráfico
import os
if os.path.exists('grafico_pine_chemicals.png'):
    files.download('grafico_pine_chemicals.png')
    print('📥 Download do gráfico iniciado')

---
## 🔧 Extras — Consultas adicionais

### Filtrar por país destino específico
```python
# Exportações de breu para Portugal
df_portugal = pd.DataFrame(todos_dados['38061000'])
df_portugal = df_portugal[df_portugal['pais'].str.contains('Portugal', case=False, na=False)]
print(df_portugal.groupby('ano')[['fob_usd','kg_liquido']].sum())
```

### Calcular preço médio anual
```python
df = pd.DataFrame(todos_dados['38061000'])
anual = df.groupby('ano').agg(fob=('fob_usd','sum'), kg=('kg_liquido','sum'))
anual['usd_kg'] = anual['fob'] / anual['kg']
print(anual)
```

### Adicionar novo NCM
```python
# Adicione na CÉLULA 2:
NCMS['38063000'] = 'Gomas ester'
# Depois re-execute as células 4 em diante
```

---
*RJ Comércio e Extração de Resinas Ltda  |  Alliance Resin Partners*